# Notebook 7 - Effect size preservation
Effect size (Cohen's d against CamCAN) of each disease before and after harmonization, on the synthetic control sites.
The reference values are the `gt_train` files written by notebook 3. The harmonized values are the train files harmonized by notebook 5. Run both notebooks first. The sites and their proportions of patients are read from their folders.
The table of |ΔSD| and the figure `effect_size_by_disease` are saved in `RESULTS/EFFECT_SIZE/effect_size_by_disease`. The figure shows, for each disease, the proportions of patients counted in at least `MIN_SITES` sites.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from joblib import Parallel, delayed
from matplotlib.patches import Patch

from src.clinical_combat.robust.robust_harmonization import (
    get_output_filename,
)
from src.clinical_combat.robust.robust_utils import (
    get_camcan_file,
    get_metrics,
    remove_covariates_effects,
)

In [ ]:
MAIN_FOLDER = "RESULTS/EFFECT_SIZE"
OUTPUT_FOLDER = f"{MAIN_FOLDER}/effect_size_by_disease"

# Settings of notebooks 3 and 5, their outputs are read here
MAE_FOLDER = "RESULTS/MAE_TEST"
BASE_NAME = "harmonized"
AUGMENTATION_COPIES = 5
AUGMENTATION_SUFFIX = (
    f"_AUG_{AUGMENTATION_COPIES}" if AUGMENTATION_COPIES else ""
)
init_harmonization_method = "pairwise"
harmonization_method = "pairwise"
SITE_SET = "ALL"  # synthetic sites drawn from all the diseases
SAMPLE_SIZE = 100  # subjects per synthetic site

# Reference values (gt_train_*) written by notebook 3
GT_ROOT = (
    f"DATA/processed/synthetic_sites/{init_harmonization_method}/"
    f"{BASE_NAME}{AUGMENTATION_SUFFIX}/{SITE_SET}"
)
# Train files harmonized by each method in notebook 5
HARM_ROOT = (
    f"{MAE_FOLDER}/PROCESS/{init_harmonization_method}/"
    f"{harmonization_method}/{SITE_SET}"
)

metrics = get_metrics()
diseases = ["AD", "MCI", "TBI", "SCHZ", "BIP", "ADHD"]

# Not white matter bundles, left out of the 45 bundles of CamCAN
DROPPED_BUNDLES = ["left_ventricle", "right_ventricle"]

# Robust method of notebook 5 (its folder name), then its label
# The order is the one of the bars and of the legend
METHODS = {
    "HC": "HC",
    "MLP_EXAMPLE": "MLP",
    "G_MAD": "G_MAD",
    "PCA_MAD": "PCA_MAD",
    "G_ZS": "G_ZS",
    "PCA_ZS": "PCA_ZS",
    "IQR": "IQR",
    "SN": "SN",
    "QN": "QN",
    "MAD": "MAD",
    "ZS": "ZS",
    "VS": "VS",
    "MMS": "MMS",
    "NO": "NO_FILTERING",
}
MIN_PATIENTS = 3  # patients of a disease needed in a site
MIN_SITES = 10  # counted sites needed to draw a proportion of patients

READ_KWARGS = {
    "usecols": [
        "sid",
        "bundle",
        "mean",
        "site",
        "age",
        "sex",
        "handedness",
        "disease",
    ],
    "dtype": {"site": str},
}
N_JOBS = -1

## Effect size functions

In [ ]:
def infer_sites(root):
    """(ratio, index) of the synthetic sites harmonized by notebook 5."""
    if not os.path.isdir(root):
        raise FileNotFoundError(f"{root} not found, run notebook 5 first.")
    sites = []
    for name in os.listdir(root):
        size, _, ratio = name.partition("_")
        path = os.path.join(root, name)
        if size != str(SAMPLE_SIZE) or not ratio.isdigit():
            continue
        if not os.path.isdir(path):
            continue
        for idx in os.listdir(path):
            if idx.isdigit() and os.path.isdir(os.path.join(path, idx)):
                sites.append((int(ratio), int(idx)))
    if not sites:
        raise ValueError(f"No synthetic site found in {root}.")
    return sorted(sites)


def gt_path(ratio, idx, metric):
    """Reference values of a site, written by notebook 3."""
    return os.path.join(
        GT_ROOT,
        f"{SAMPLE_SIZE}_{ratio}",
        str(idx),
        f"gt_train_{SAMPLE_SIZE}_{ratio}_{idx}_{metric}.csv",
    )


def train_path(ratio, idx, metric):
    """Train file of a site, as cleaned by notebook 5."""
    return os.path.join(
        HARM_ROOT,
        f"{SAMPLE_SIZE}_{ratio}",
        str(idx),
        metric,
        f"train_{SAMPLE_SIZE}_{ratio}_{idx}_{metric}.csv",
    )


def harm_path(ratio, idx, metric, robust_method):
    """Train file of a site harmonized by one method in notebook 5."""
    train_file = train_path(ratio, idx, metric)
    return os.path.join(
        os.path.dirname(train_file),
        robust_method,
        get_output_filename(
            train_file, metric, harmonization_method, robust_method
        ),
    )


def stats(df, by):
    """Number, mean and sample SD (ddof = 1) of the values."""
    return df.groupby(by)["mean_no_cov"].agg(n="size", mean="mean", sd="std")


def cohen_d(values, cam):
    """Cohen's d (pooled SD) of each disease against CamCAN, per bundle."""
    # The covariate model is refitted on the site values and CamCAN together
    fit = remove_covariates_effects(
        pd.concat(
            [values.assign(origin="site"), cam.assign(origin="camcan")],
            ignore_index=True,
        )
    )
    patients = fit[(fit["origin"] == "site") & fit["disease"].isin(diseases)]
    g = stats(patients, ["bundle", "disease"]).join(
        stats(fit[fit["origin"] == "camcan"], "bundle"),
        on="bundle",
        rsuffix="_cam",
    )
    pooled_sd = np.sqrt(
        ((g["n"] - 1) * g["sd"] ** 2 + (g["n_cam"] - 1) * g["sd_cam"] ** 2)
        / (g["n"] + g["n_cam"] - 2)
    )
    return pd.DataFrame(
        {"n_patients": g["n"], "d": (g["mean"] - g["mean_cam"]) / pooled_sd}
    )


def site_rows(ratio, idx):
    """d_ref and d_harm of every row of one synthetic control site.

    Returns the rows and the files not found. A site without its reference
    values or its train file is left out. A method without its file for every
    metric is left out of the site.
    """
    out, missing = [], []
    for metric in metrics:
        gt_file = gt_path(ratio, idx, metric)
        absent = [
            path
            for path in (gt_file, train_path(ratio, idx, metric))
            if not os.path.isfile(path)
        ]
        if absent:
            return pd.DataFrame(), absent
        cam = pd.read_csv(get_camcan_file(metric, cleaned=True), **READ_KWARGS)
        cam = cam[~cam["bundle"].isin(DROPPED_BUNDLES)]
        d_ref = cohen_d(pd.read_csv(gt_file, **READ_KWARGS), cam)
        for folder, method in METHODS.items():
            harm_file = harm_path(ratio, idx, metric, folder)
            if not os.path.isfile(harm_file):
                missing.append((method, harm_file))
                continue
            harm_df = pd.read_csv(harm_file, **READ_KWARGS)
            d_harm = cohen_d(harm_df, cam)
            g = d_ref.rename(columns={"d": "d_ref"}).assign(d_harm=d_harm["d"])
            g = g[g["n_patients"] >= MIN_PATIENTS].reset_index()
            out.append(
                (
                    method,
                    g.assign(
                        ratio=ratio, site_id=idx, metric=metric, method=method
                    ),
                )
            )
    left_out = {method for method, _ in missing}
    frames = [g for method, g in out if method not in left_out]
    rows = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    return rows, [path for _, path in missing]

## Run the calculations

In [ ]:
sites = infer_sites(HARM_ROOT)
results = Parallel(n_jobs=N_JOBS)(
    delayed(site_rows)(ratio, idx) for ratio, idx in sites
)
frames = [site for site, _ in results if len(site)]
missing = [path for _, paths in results for path in paths]
if missing:
    print(f"{len(missing)} files not found, for example {missing[0]}")
if not frames:
    raise ValueError("No site could be read, run notebooks 3 and 5 first.")
rows = pd.concat(frames, ignore_index=True)
print(f"{len(sites)} sites, {len(rows):,} rows")

## Table and figure

In [ ]:
rows["abs_err"] = (rows["d_harm"] - rows["d_ref"]).abs()  # |ΔSD|

# Same rows for every counted site, so the flat mean is the mean of site means
assert (
    rows.groupby(["ratio", "site_id", "disease", "method"]).size().nunique()
    == 1
)

table = (
    rows.groupby(["ratio", "disease", "method"])
    .agg(abs_delta_sd=("abs_err", "mean"), n_sites=("site_id", "nunique"))
    .reset_index()
)

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
table_path = os.path.join(OUTPUT_FOLDER, "abs_delta_sd_by_disease.csv")
table.to_csv(table_path, index=False)
print(f"{len(table)} rows written in {table_path}")

# |ΔSD| at 80 % of patients for AD and MCI
at80 = table[
    (table["ratio"] == 80)
    & table["disease"].isin(["AD", "MCI"])
    & table["method"].isin(["HC", "MLP", "NO_FILTERING"])
]
display(
    at80[["disease", "method", "abs_delta_sd", "n_sites"]]
    .round(4)
    .reset_index(drop=True)
)

In [ ]:
labels = list(METHODS.values())
# HC, MLP, the 11 other methods in grey from dark to light, NO_FILTERING
grey = plt.cm.gray(np.linspace(0.05, 0.95, len(labels) - 3))
colors = dict(zip(labels, ["chartreuse", "yellow", *grey, "red"]))
order = [method for method in labels if method in set(table["method"])]

# Only the proportions of patients counted in enough sites are drawn
shown = table[table["n_sites"] >= MIN_SITES]
means = shown.groupby(["disease", "ratio", "method"])["abs_delta_sd"].mean()

fig, axes = plt.subplots(2, 3, figsize=(17, 9.5), sharey=True)
for ax, disease in zip(axes.flat, diseases):
    ratios = sorted(shown.loc[shown["disease"] == disease, "ratio"].unique())
    # One group of bars per ratio, one bar per method
    x, width = np.arange(len(ratios)), 0.8 / len(order)
    for k, method in enumerate(order):
        ax.bar(
            x + (k - len(order) / 2 + 0.5) * width,
            [means.get((disease, ratio, method), np.nan) for ratio in ratios],
            width,
            color=colors[method],
            edgecolor="black",
            linewidth=0.5,
        )
    if not ratios:
        ax.text(
            0.5,
            0.5,
            f"No proportion with at least {MIN_SITES} sites",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
    ax.set_xticks(x, [f"{ratio}%" for ratio in ratios], fontsize=12)
    ax.set_title(disease, fontsize=13)
    ax.set_ylim(0, 0.6)
    ax.tick_params(labelsize=11)
for ax in axes[:, 0]:
    ax.set_ylabel("|Δ SD|", fontsize=13)
for ax in axes[1]:
    ax.set_xlabel("Proportion of patients (%)", fontsize=13)
fig.legend(
    handles=[
        Patch(
            facecolor=colors[method],
            edgecolor="black",
            linewidth=0.5,
            label=method,
        )
        for method in order
    ],
    loc="center right",
    fontsize=13,
    frameon=False,
)
fig.tight_layout(rect=[0, 0, 0.88, 1])
fig.savefig(os.path.join(OUTPUT_FOLDER, "effect_size_by_disease.png"), dpi=150)
plt.show()